Generating the 4 raw CSV files (customers, products, orders, order_items) with intentional data issues.

In [0]:
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta
from faker import Faker

fake=Faker()
random.seed(42)
np.random.seed(42)
Faker.seed(42)

BASE_DIR="/Volumes/assignment8/assignment8schema/assignment8volume"
os.makedirs(BASE_DIR, exist_ok=True)

N_CUSTOMERS=600
N_PRODUCTS=150
N_ORDERS=3000
N_ORDER_ITEMS=7000

print("Working directory:", BASE_DIR)

Working directory: /Volumes/assignment8/assignment8schema/assignment8volume


In [0]:
def generate_customers(n=N_CUSTOMERS):
    rows=[]
    customer_types=["REGULAR", "PREMIUM", "VIP"]
    type_weights=[0.7, 0.22, 0.08]

    start_reg=datetime(2023, 1, 1)
    end_reg=datetime(2026, 6, 1)

    for cid in range(1, n+1):
        name=fake.name()
        if random.random()<0.02:
            bad_type=random.choice(["no_at", "no_domain"])
            local=name.lower().replace(" ", ".")
            if bad_type=="no_at":
                email=f"{local}example.com"
            else:
                email=f"{local}@"
        else:
            email=fake.email()

        reg_date=start_reg+timedelta(
            days=random.randint(0, (end_reg-start_reg).days)
        )

        ctype=random.choices(customer_types, weights=type_weights, k=1)[0]

        rows.append({
            "customer_id": cid,
            "customer_name": name,
            "email": email,
            "registration_date": reg_date.strftime("%Y-%m-%d %H:%M:%S"),
            "customer_type": ctype,
        })

    df=pd.DataFrame(rows)
    df.to_csv(f"{BASE_DIR}/customers.csv", index=False)
    return df

customers_df=generate_customers()
print(customers_df.shape)
customers_df.head()

(600, 5)


,customer_id,customer_name,email,registration_date,customer_type
0,1,Allison Hill,donaldgarcia@example.net,2023-10-13 00:00:00,PREMIUM
1,2,Angie Henderson,davisjesse@example.net,2026-01-21 00:00:00,REGULAR
2,3,Cristian Santos,lrobinson@example.com,2023-03-03 00:00:00,REGULAR
3,4,Abigail Shaffer,jpeterson@example.org,2026-05-17 00:00:00,REGULAR
4,5,Gabrielle Davis,howardmaurice@example.com,2026-01-21 00:00:00,REGULAR


In [0]:
def generate_products(n=N_PRODUCTS):
    categories={
        "Electronics": ["Mobiles", "Laptops", "Accessories", "Audio"],
        "Clothing": ["Men", "Women", "Kids", "Footwear"],
        "Home": ["Furniture", "Kitchen", "Decor", "Appliances"],
        "Books": ["Fiction", "Non-Fiction", "Academic", "Comics"],
    }

    rows=[]
    for pid in range(1, n+1):
        category=random.choice(list(categories.keys()))
        subcategory=random.choice(categories[category])
        base_name=fake.word().capitalize()+" "+subcategory[:-1] if subcategory.endswith("s") else fake.word().capitalize()
        product_name=f"{base_name} {fake.word().capitalize()}"

        if random.random()<0.25:
            variant=random.choice(["upper", "lower", "spaces"])
            if variant=="upper":
                product_name=product_name.upper()
            elif variant=="lower":
                product_name=product_name.lower()
            else:
                product_name=f"   {product_name}   "

        cost_price=round(random.uniform(50, 5000), 2)

        rows.append({
            "product_id": pid,
            "product_name": product_name,
            "category": category,
            "subcategory": subcategory,
            "cost_price": cost_price,
        })

    df=pd.DataFrame(rows)
    df.to_csv(f"{BASE_DIR}/products.csv", index=False)
    return df

products_df=generate_products()
print(products_df.shape)
products_df.head()

(150, 5)


,product_id,product_name,category,subcategory,cost_price
0,1,Notice Kid Their,Clothing,Kids,1626.72
1,2,Ability Watch,Books,Academic,2561.21
2,3,Writer Stage,Books,Academic,132.35
3,4,Resource Range,Home,Furniture,2939.33
4,5,Lawyer Nearly,Electronics,Audio,3295.18


In [0]:
def generate_orders(n=N_ORDERS, customers_df=customers_df):
    statuses=["PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"]
    status_weights=[0.15, 0.15, 0.55, 0.1, 0.05]
    regions=["NORTH", "SOUTH", "EAST", "WEST", "CENTRAL"]

    valid_customer_ids=customers_df["customer_id"].tolist()

    start_date=datetime(2024, 1, 1)
    end_date=datetime(2026, 7, 1)

    rows=[]
    for oid in range(1, n+1):
        if random.random()<0.05:
            customer_id=""
        else:
            customer_id=random.choice(valid_customer_ids)

        order_dt=start_date+timedelta(
            seconds=random.randint(0, int((end_date-start_date).total_seconds()))
        )

        if random.random()<0.08:
            order_date_str=order_dt.strftime("%d-%m-%Y")
        else:
            order_date_str=order_dt.strftime("%Y-%m-%d %H:%M:%S")

        status=random.choices(statuses, weights=status_weights, k=1)[0]
        region_code=random.choice(regions)

        rows.append({
            "order_id": oid,
            "customer_id": customer_id,
            "order_date": order_date_str,
            "status": status,
            "region_code": region_code,
        })

    df=pd.DataFrame(rows)
    df.to_csv(f"{BASE_DIR}/orders.csv", index=False)
    return df

orders_df=generate_orders()
print(orders_df.shape)
orders_df.head()

(3000, 5)


,order_id,customer_id,order_date,status,region_code
0,1,26,2026-01-19 23:00:38,DELIVERED,NORTH
1,2,146,2025-06-25 02:38:00,DELIVERED,SOUTH
2,3,357,2026-01-09 16:37:33,DELIVERED,WEST
3,4,308,2024-03-10 09:54:17,PLACED,EAST
4,5,169,2025-01-13 21:44:01,DELIVERED,EAST


In [0]:
def generate_order_items(n=N_ORDER_ITEMS, orders_df=orders_df, products_df=products_df):
    valid_order_ids=orders_df["order_id"].tolist()
    valid_product_ids=products_df["product_id"].tolist()
    product_price_map=dict(zip(products_df["product_id"], products_df["cost_price"]))

    rows=[]
    for item_id in range(1, n+1):
        order_id=random.choice(valid_order_ids)
        product_id=random.choice(valid_product_ids)

        base_price=product_price_map[product_id]
        unit_price=round(base_price*random.uniform(1.1, 1.8), 2)

        quantity=random.randint(1, 5)
        if random.random()<0.03:
            quantity=-quantity

        discount_percent=round(random.uniform(0, 40), 1)

        rows.append({
            "item_id": item_id,
            "order_id": order_id,
            "product_id": product_id,
            "quantity": quantity,
            "unit_price": unit_price,
            "discount_percent": discount_percent,
        })

    rows.append({
        "item_id": n+1,
        "order_id": max(valid_order_ids)+9999,
        "product_id": random.choice(valid_product_ids),
        "quantity": 2,
        "unit_price": 500.0,
        "discount_percent": 10.0,
    })
    rows.append({
        "item_id": n+2,
        "order_id": random.choice(valid_order_ids),
        "product_id": random.choice(valid_product_ids),
        "quantity": 1,
        "unit_price": 300.0,
        "discount_percent": 150.0,
    })
    rows.append({
        "item_id": n+3,
        "order_id": random.choice(valid_order_ids),
        "product_id": random.choice(valid_product_ids),
        "quantity": 0,
        "unit_price": 100.0,
        "discount_percent": 5.0,
    })

    df=pd.DataFrame(rows)
    df.to_csv(f"{BASE_DIR}/order_items.csv", index=False)
    return df

order_items_df=generate_order_items()
print(order_items_df.shape)
order_items_df.head()

(7003, 6)


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,1,1890,31,3,1312.02,39.2
1,2,1882,52,5,2962.83,13.3
2,3,2001,131,2,6963.13,21.2
3,4,1726,22,1,5057.73,9.6
4,5,2640,134,2,7122.58,1.5
